# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maheen-armghan/flyrank-internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


**Lane: Refresh / Content Opportunity Scoring.**

I'm choosing this lane because it directly extends the exploration I already did in the Week 1 discovery notebooks —
testing beliefs about position, CTR, content age, and trend direction on the starter dataset. This lane turns that
exploration into something actionable: a ranked queue of pages a reviewer should look at first for refresh, expansion,
or pruning. The starter dataset's current label (`trend_direction == "down"`) is a proxy based on the current window
rather than a true future outcome, so a stronger version of this question I'd want to move toward is: given a page's
signals over the prior 90 days, does it decline or recover over the next 30 days?

## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


**Decision this improves:** which pages a content reviewer should prioritize checking first, out of thousands, given limited review capacity each week.

**Who acts on it:** a content strategist or SEO reviewer with a fixed weekly capacity (e.g. can only manually review ~50 pages), who needs a ranked, evidence-backed list instead of scanning the whole site.

**Cost of a wrong call:**
- **False positive** (flagging a healthy page as needing review): wastes a reviewer's limited time — capacity spent on a page that didn't need it, while a genuinely declining page waits.
- **False negative** (missing a page that's actually declining): a real problem goes unnoticed until it's worse — lost traffic/visibility compounds the longer it's unaddressed.

Because review capacity is limited, this is a ranking/prioritization problem, not a pass/fail one — the goal is ordering candidates so the highest-value review happens first, not proving any single page will recover if touched.

## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [3]:
import os

REPO_URL = "https://github.com/maheen-armghan/flyrank-internship.git"
REPO_NAME = "flyrank-internship"

if not os.path.exists(REPO_NAME):
    !git clone {REPO_URL}

os.chdir(REPO_NAME)
print("Now in:", os.getcwd())

Cloning into 'flyrank-internship'...
remote: Enumerating objects: 323, done.
remote: Counting objects: 100% (323/323), done.
remote: Compressing objects: 100% (139/139), done.
remote: Total 323 (delta 173), reused 295 (delta 156), pack-reused 0 (from 0)
Receiving objects: 100% (323/323), 1.91 MiB | 10.98 MiB/s, done.
Resolving deltas: 100% (173/173), done.
Now in: /content/flyrank-internship


In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# match the reference pipeline's own filters (see docs/ml-intern-dataset-and-lane-guide.md, Section 5)
df = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)]
df = df.drop_duplicates(subset="content_id")

label = (df["trend_direction"] == "down").astype(int)

print("Rows after filtering:", len(df))
print("Declining share: {:.1%}".format(label.mean()))
print()
print("Median days_since_last_update by decline status:")
print(df.groupby(label)["days_since_last_update"].median())

Rows after filtering: 30000
Declining share: 54.2%

Median days_since_last_update by decline status:
trend_direction
0    20.0
1    20.0
Name: days_since_last_update, dtype: float64


In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Loaded `content_refresh_anonymized.csv` and applied the same filters the reference pipeline uses (`impressions_90d > 0`, `content_age_days >= 90`, deduplicated by `content_id`). After filtering: **30,000 rows** remain. Of those, **54.2%** are currently labeled declining (`trend_direction == "down"`) — a strong, workable base rate; not too rare to model, not so imbalanced it's trivial.

Interestingly, median `days_since_last_update` is **identical (20.0 days) for both declining and non-declining pages**. This means staleness alone does not obviously separate the two groups at the median — an initial belief worth testing ("stale pages decline more") doesn't hold up this simply. This is an early, honest signal that decline is likely driven by a combination of factors rather than staleness on its own, which supports the case for a model over a single hand-written rule.

## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


**What this work CAN say:** which pages, based on observed signals, look most similar to pages that have historically declined — an observed, directional pattern that supports a human reviewer's prioritization decision. Results are decision-support: they help a reviewer allocate limited attention, not a guarantee of what will happen to any individual page.

**What this work CANNOT say:** that any specific page is guaranteed to decline or recover; that refreshing a page will cause it to recover (that would require a controlled experiment, not observational data); that any finding reveals how Google's ranking algorithm actually works; and it cannot claim causal proof of any kind from correlational or observed patterns alone.

I'll consistently use language like "observed," "associated with," "directional signal," and "decision-support" — and avoid words like "proves," "causes," or "predicts Google's algorithm."

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.